In [1]:
import datetime

import colormaps
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import xarray as xr
from jetutils.anyspell import get_persistent_jet_spells, mask_from_spells_pl, subset_around_onset, get_persistent_spell_times_from_som, get_spells, extend_spells, gb_index, make_daily
from jetutils.clustering import Experiment, RAW_REALSPACE, labels_to_centers
from jetutils.data import *
from jetutils.geospatial import *
from jetutils.definitions import *
from jetutils.jet_finding import *
from jetutils.plots import *
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from tqdm import tqdm
import polars.selectors as cs

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

IPython could not be loaded!


# data

In [ ]:
import datetime
import os
from itertools import pairwise
from pathlib import Path
from jetutils.definitions import DATADIR, compute
from jetutils.data import standardize, extract
import numpy as np 
import xarray as xr
from tqdm import tqdm
from urllib.request import urlretrieve

experiment_dict = {
    "past": "BHISTcmip6",
    "future": "BSSP370cmip6",
}
yearbounds = {
    "past": np.arange(1970, 2011, 10),
    "future": np.arange(2055, 2106, 10),
}
levels_dict = {
    "high_wind": list(range(14, 18)),
    "mid_wind": 20,
    "PRECL": "all",
    "TS": "all",
}
# yearbounds["past"][-1] = yearbounds["past"][-1] - 5
yearbounds["future"][-1] = yearbounds["future"][-1] - 4
timebounds = {key: [f"{year1}0101-{year2 - 1}1231" for year1, year2 in pairwise(val)] for key, val in yearbounds.items()}
years = {key: [list(range(max(year1, 2060) if key == "future" else year1, year2)) for year1, year2 in pairwise(val)] for key, val in yearbounds.items()}

members = [f"{year}.{str(number).zfill(3)}" for year, number in zip(range(1001, 1201, 20), range(1, 11))]
for startyear in [1231, 1251, 1281, 1301]:
    members.extend(f"{startyear}.{str(number).zfill(3)}" for number in range(1, 11))
    
members2 = [f"r{number}i{year}p1f1" for year, number in zip(range(1001, 1201, 20), range(1, 11))]
for startyear in [1231, 1251, 1281, 1301]:
    members2.extend(f"r{number}i{startyear}p1f1" for number in range(1, 11))
    
season = None
minlon = -80
maxlon = 40
minlat = 15
maxlat = 80
    
def get_url(varname: str, period: str, member: str, timebounds: str, minlon: float, maxlon: float, minlat: float, maxlat: float, year: int):
    experiment = experiment_dict[period]
    h = 6 if varname in ["U", "V", "T"] else 1

    base_url = f"https://tds.gdex.ucar.edu/thredds/ncss/grid/files/d651056/CESM2-LE/atm/proc/tseries/day_1/{varname}/b.e21.{experiment}.f09_g17.LE2-{member}.cam.h{h}.{varname}.{timebounds}.nc"
    specs = f"var={varname}&north={maxlat}&west={minlon}&east={maxlon}&south={minlat}&horizStride=1&time_start={year}-01-01T00:00:00Z&time_end={year}-12-31T00:00:00Z&&&accept=netcdf4ext"
    return f"{base_url}?{specs}"

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)
        
        
def download_url(url, output_path):
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=url.split('/')[-1]) as t:
        urlretrieve(url, filename=output_path, reporthook=t.update_to)


basepath = Path(f"{DATADIR}/CESM2/high_wind/ssp370")
scratchdir = Path("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/ssp370/raw")
var = "T"
period = "future"

for member1, member2 in zip(members, members2):
    for years_, tb in zip(years["future"], timebounds["future"]):
        for year in years_:
            url = get_url(var, period, member1, tb, minlon, maxlon, minlat, maxlat, year)
            scratchpath = scratchdir.joinpath(f"{member2}-{year}.nc")
            if scratchpath.is_file():
                continue
            download_url(url, scratchpath)

In [33]:
for member1, member2 in zip(members, members2):
    for years_, tb in zip(years["future"], timebounds["future"]):
        for year in years_:
            url = get_url(var, period, member1, tb, minlon, maxlon, minlat, maxlat, year)
            scratchpath = scratchdir.joinpath(f"{member2}-{year}.nc")
            if scratchpath.is_file():
                continue
            download_url(url, scratchpath)
            # da_ = xr.open_dataset(scratchpath)[var]
            # da_ = standardize(da_)
            # da_ = extract(da_, "all", None, minlon, maxlon, minlat, maxlat, levels=levels_dict["high_wind"])

b.e21.BSSP370cmip6.f09_g17.LE2-1001.001.cam.h6.T.20550101-20641231.nc?var=T&north=80&west=-80&east=40&south=15&horizStride=1&time_start=2060-01-01T00:00:00Z&time_end=2060-12-31T00:00:00Z&&&accept=netcdf4-classic: 473MB [01:07, 6.98MB/s]                               
b.e21.BSSP370cmip6.f09_g17.LE2-1001.001.cam.h6.T.20550101-20641231.nc?var=T&north=80&west=-80&east=40&south=15&horizStride=1&time_start=2061-01-01T00:00:00Z&time_end=2061-12-31T00:00:00Z&&&accept=netcdf4-classic: 473MB [03:13, 2.45MB/s]                                
b.e21.BSSP370cmip6.f09_g17.LE2-1001.001.cam.h6.T.20550101-20641231.nc?var=T&north=80&west=-80&east=40&south=15&horizStride=1&time_start=2062-01-01T00:00:00Z&time_end=2062-12-31T00:00:00Z&&&accept=netcdf4-classic: 473MB [01:00, 7.80MB/s]                               
b.e21.BSSP370cmip6.f09_g17.LE2-1001.001.cam.h6.T.20550101-20641231.nc?var=T&north=80&west=-80&east=40&south=15&horizStride=1&time_start=2063-01-01T00:00:00Z&time_end=2063-12-31T00:00:00Z&&&accept

KeyboardInterrupt: 

In [ ]:
for member1, member2 in zip(members, members2):
    opath = basepath.joinpath(f"{member2}.nc")
    if opath.is_file():
        print(member1, "aka", member2, "already there")
        continue
    scratchpaths = []
    da = []
    for years_, tb in zip(years["future"], timebounds["future"]):
        for year in years_:
            url = get_url(var, period, member1, tb, minlon, maxlon, minlat, maxlat, year)
            scratchpath = scratchdir.joinpath(url.split("/")[-1].split("?")[0])
            scratchpaths.append(scratchpath)
            if scratchpath.is_file():
                continue
            download_url(url, scratchpath)
            da_ = xr.open_dataset(scratchpath)[var]
            da_ = standardize(da_)
            da_ = extract(da_, "all", None, minlon, maxlon, minlat, maxlat, levels=levels_dict["high_wind"])
    #         da.append(da_)
    # da = xr.concat(da, "time")
    # ds = xr.open_mfdataset(basepath.glob(f"{member2}-????.nc"))

    # for coord in ["lon", "lat", "lev"]:
    #     da[coord] = da[coord].astype(np.float32)

    # da["time"] = da.indexes["time"].to_datetimeindex(time_unit="us") + datetime.timedelta(hours=12)
    # ds["time"] = ds.indexes["time"].to_datetimeindex(time_unit="us")
    # da = da.sel(time=ds.time.values, lon=ds.lon.values, lat=ds.lat.values)
    # da = compute(da.sel(lev=ds["lev"]), progress_flag=True)
    # da.to_netcdf(opath)
    # for scratchpath in scratchpaths:
    #     os.remove(scratchpath)
    # print(member1, "aka", member2, "done")
    # del da, ds

1001.001 aka r1i1001p1f1 already there
1021.002 aka r2i1021p1f1 already there
1041.003 aka r3i1041p1f1 already there
1061.004 aka r4i1061p1f1 already there
1081.005 aka r5i1081p1f1 already there
1101.006 aka r6i1101p1f1 already there
1121.007 aka r7i1121p1f1 already there
1141.008 aka r8i1141p1f1 already there
1161.009 aka r9i1161p1f1 already there
1181.010 aka r10i1181p1f1 already there
1231.001 aka r1i1231p1f1 already there
1231.002 aka r2i1231p1f1 already there
1231.003 aka r3i1231p1f1 already there
1231.004 aka r4i1231p1f1 already there
1231.005 aka r5i1231p1f1 already there
1231.006 aka r6i1231p1f1 already there
1231.007 aka r7i1231p1f1 already there
1231.008 aka r8i1231p1f1 already there


b.e21.BSSP370cmip6.f09_g17.LE2-1231.009.cam.h6.T.20450101-20541231.nc?var=T&north=90&west=-180&east=180&south=0&horizStride=1&time_start=2045-01-01T00:00:00Z&time_end=2045-12-31T00:00:00Z&&&accept=netcdf4-classic: 0.00B [00:00, ?B/s]

b.e21.BSSP370cmip6.f09_g17.LE2-1231.009.cam.h6.T.20450101-20541231.nc?var=T&north=90&west=-180&east=180&south=0&horizStride=1&time_start=2045-01-01T00:00:00Z&time_end=2045-12-31T00:00:00Z&&&accept=netcdf4-classic: 1.90GB [02:40, 11.8MB/s]                                


RuntimeError: Unspecified error in H5DSget_num_scales (return value <0)

In [17]:
xr.open_dataset(scratchpath)

<xarray.Dataset> Size: 3GB
Dimensions:  (time: 365, lev: 32, lat: 96, lon: 288)
Coordinates:
  * time     (time) object 3kB 2045-01-01 00:00:00 ... 2045-12-31 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lat      (lat) float64 768B 0.4712 1.414 2.356 3.298 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Data variables:
    T        (time, lev, lat, lon) float64 3GB ...
Attributes: (12/14)
    Conventions:         CF-1.0
    source:              CAM
    case:                b.e21.BSSP370cmip6.f09_g17.LE2-1231.009
    logname:             sunseon
    host:                mom2
    initial_file:        b.e21.BHISTcmip6.f09_g17.LE2-1231.009.cam.i.2015-01-...
    ...                  ...
    time_period_freq:    day_1
    History:             Translated to CF-1.0 Conventions by Netcdf-Java CDM ...
    geospatial_lat_min:  3.0531133177191805e-15
    geospatial_lat_max:  90.0
    geospatial_lon_min:  -180.0
    geospatial_lon_max:  180.0

In [43]:
download_url("https://tds.gdex.ucar.edu/thredds/fileServer/files/d651056/CESM2-LE/atm/proc/tseries/day_1/T/b.e21.BSSP370cmip6.f09_g17.LE2-1231.009.cam.h6.T.20450101-20541231.nc", scratchpath)

b.e21.BSSP370cmip6.f09_g17.LE2-1231.009.cam.h6.T.20450101-20541231.nc:   0%|          | 66.8M/37.6G [00:03<37:09, 16.9MB/s]    


KeyboardInterrupt: 

In [31]:
url

'https://tds.ucar.edu/thredds/fileServer/datazone/campaign/cgd/cesm/CESM2-LE/atm/proc/tseries/day_1/T/b.e21.BSSP370cmip6.f09_g17.LE2-1231.009.cam.h6.T.20950101-21001231.nc?api-token=ayhBFVYTOtGi2LM2cHDn6DjFCoKeCAqt69z8Ezt4#mode=bytes'

In [30]:
scratchpath

PosixPath('/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/ssp370/raw/b.e21.BSSP370cmip6.f09_g17.LE2-1231.009.cam.h6.T.20950101-21001231.nc')

In [29]:
xr.open_dataset(scratchpath, engine="h5netcdf")[var]

OSError: Unable to synchronously open file (file signature not found)

In [27]:
url

'https://tds.ucar.edu/thredds/fileServer/datazone/campaign/cgd/cesm/CESM2-LE/atm/proc/tseries/day_1/T/b.e21.BSSP370cmip6.f09_g17.LE2-1231.009.cam.h6.T.20950101-21001231.nc?api-token=ayhBFVYTOtGi2LM2cHDn6DjFCoKeCAqt69z8Ezt4#mode=bytes'

# rest

In [2]:
ds_cesm = xr.open_dataset("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/historical/ds.zarr", engine="zarr")
ds_cesm = ds_cesm.reset_coords("time_bnds", drop=True).drop_dims("nbnd")
ds_cesm = ds_cesm.chunk({"member": 1, "time": 100, "lat": -1, "lon": -1})
ds_cesm["theta"] = (ds_cesm["t"] * (1000 / ds_cesm["lev"]) ** KAPPA).astype(np.float32)
ds_cesm = ds_cesm.drop_vars("t")
dh = DataHandler.from_basepath_and_da("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/high_wind/historical/results", ds_cesm)
exp = JetFindingExperiment(dh)
all_jets_one_df = exp.find_jets()
props_as_df = exp.props_as_df()

In [ ]:
both_jets = {}
both_paths = {}
for run in ["historical", "ssp370"]:
    dh = DataHandler.from_specs("Henrik_data", run, ("high_wind", ["u", "v", "s", "theta"]), "6H", minlon=-80, maxlon=40, minlat=15, maxlat=80, levels=20000)
    exp = JetFindingExperiment(dh)
    all_jets_one_df = exp.find_jets(force=False, n_coarsen=1, smooth_s=5, alignment_thresh=0.55, base_s_thresh=0.55, int_thresh_factor=0.2, hole_size=5)
    if "is_polar" not in all_jets_one_df.columns:
        theta300 = open_da("Henrik_data", run, ("high_wind", ["s", "theta"]), "6H", minlon=-80, maxlon=40, minlat=15, maxlat=80, levels=30000).rename({"s": "s300", "theta": "theta300"})
        all_jets_one_df = add_feature_for_cat(all_jets_one_df, "s300", theta300, ofile_ajdf=exp.path.joinpath("all_jets_one_df.parquet"), force=False)
        all_jets_one_df = add_feature_for_cat(all_jets_one_df, "theta300", theta300, ofile_ajdf=exp.path.joinpath("all_jets_one_df.parquet"), force=False)
        all_jets_one_df = exp.categorize_jets(None, ["s300", "theta300"], force=False, n_init=15, init_params="k-means++", mode="month").cast({"time": pl.Datetime("ms")})
    phat_jets = all_jets_one_df.filter((pl.col("is_polar").mean().over(["time", "jet ID"]) < 0.5) | ((pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5) & (pl.col("int").mode().first().over(["time", "jet ID"]) > 1.e8)))
    both_paths[run] = exp.path
    both_jets[run] = phat_jets.with_columns(**{"jet ID": (pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5).cast(pl.UInt32())})

In [3]:
cross_all = track_jets(all_jets_one_df)

100%|██████████| 49/49 [14:20<00:00, 17.56s/it]


In [9]:
cross_all

member,time,jet ID,s,theta,is_polar,time_right,jet ID_right,s_right,theta_right,is_polar_right,dist,lon_overlap,ds,dtheta,dis_polar
str,datetime[ms],u32,f32,f32,f64,datetime[ms],u32,f32,f32,f64,f64,f64,f32,f32,f64
"""r6i1251p1f1""",1970-01-01 12:00:00,0,41.378868,344.171448,0.006078,1970-01-02 12:00:00,0,56.771423,347.165405,0.006939,3.7866e6,0.3,15.392555,2.993958,0.000861
"""r6i1251p1f1""",1970-01-01 12:00:00,0,41.378868,344.171448,0.006078,1970-01-02 12:00:00,1,60.971439,336.349121,0.477548,1.0403e7,0.0,19.592571,7.822327,0.47147
"""r6i1251p1f1""",1970-01-01 12:00:00,0,41.378868,344.171448,0.006078,1970-01-02 12:00:00,2,52.765053,317.104828,0.999998,8.6265e6,0.172414,11.386185,27.06662,0.99392
"""r6i1251p1f1""",1970-01-01 12:00:00,1,62.876019,342.632385,0.055807,1970-01-02 12:00:00,0,56.771423,347.165405,0.006939,8.4673e6,0.0,6.104595,4.53302,0.048868
"""r6i1251p1f1""",1970-01-01 12:00:00,1,62.876019,342.632385,0.055807,1970-01-02 12:00:00,1,60.971439,336.349121,0.477548,1.7860e6,0.673077,1.904579,6.283264,0.421741
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""r4i1281p1f1""",2009-12-30 12:00:00,1,48.487206,343.877045,0.033009,2009-12-31 12:00:00,1,53.691483,348.404602,0.030272,1.4916e6,0.666667,5.204277,4.527557,0.002738
"""r4i1281p1f1""",2009-12-30 12:00:00,1,48.487206,343.877045,0.033009,2009-12-31 12:00:00,2,52.380051,324.615356,0.997621,9.9812e6,0.106195,3.892845,19.261688,0.964611
"""r4i1281p1f1""",2009-12-30 12:00:00,2,54.530598,324.35144,0.995748,2009-12-31 12:00:00,0,39.940052,346.954987,0.004139,8.0230e6,0.727273,14.590546,22.603546,0.991609


In [7]:
pers_from_cross_catd(cross_all)

member,time,jet ID,s,theta,is_polar,time_right,jet ID_right,s_right,theta_right,is_polar_right,dist,lon_overlap,ds,dtheta,dis_polar,spell_of,pers
str,datetime[ms],u32,f32,f32,f64,datetime[ms],u32,f32,f32,f64,f64,f64,f32,f32,f64,str,f64
"""r6i1251p1f1""",1970-01-01 12:00:00,0,41.378868,344.171448,0.006078,1970-01-02 12:00:00,0,56.771423,347.165405,0.006939,3.7866e6,0.3,15.392555,2.993958,0.000861,"""STJ""",0.0
"""r6i1251p1f1""",1970-01-01 12:00:00,1,62.876019,342.632385,0.055807,1970-01-02 12:00:00,1,60.971439,336.349121,0.477548,1.7860e6,0.673077,1.904579,6.283264,0.421741,"""EDJ""",0.0
"""r6i1251p1f1""",1970-01-01 12:00:00,2,56.661385,317.51889,0.999998,1970-01-02 12:00:00,2,52.765053,317.104828,0.999998,9.2045e6,0.258621,3.896332,0.4140625,1.8990e-7,"""EDJ""",0.0
"""r6i1251p1f1""",1970-01-02 12:00:00,0,56.771423,347.165405,0.006939,1970-01-03 12:00:00,0,62.47179,346.73877,0.009832,2.1151e6,0.6,5.700367,0.426636,0.002893,"""STJ""",0.0
"""r6i1251p1f1""",1970-01-02 12:00:00,1,60.971439,336.349121,0.477548,1970-01-03 12:00:00,1,57.682125,336.889221,0.472302,2.5014e6,0.666667,3.289314,0.5401,0.005246,"""EDJ""",0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""r4i1281p1f1""",2009-12-29 12:00:00,1,43.822239,343.125214,0.023026,2009-12-30 12:00:00,1,48.487206,343.877045,0.033009,2.4964e6,0.7,4.664967,0.751831,0.009984,"""EDJ""",0.0
"""r4i1281p1f1""",2009-12-29 12:00:00,2,54.293438,325.654175,0.993475,2009-12-30 12:00:00,2,54.530598,324.35144,0.995748,8.9155e6,0.672566,0.23716,1.302734,0.002273,"""EDJ""",0.0
"""r4i1281p1f1""",2009-12-30 12:00:00,0,42.6082,348.59613,0.002478,2009-12-31 12:00:00,0,39.940052,346.954987,0.004139,2.2294e6,0.727273,2.668148,1.641144,0.001661,"""STJ""",0.0
